# Communication Gap — CraigslistBargains Evaluation

Measures the **Communication Gap (CG)** metric across buyer→seller negotiation exchanges
from the [Stanford CraigslistBargains dataset](https://huggingface.co/datasets/stanfordnlp/craigslist_bargains).

## What we measure

For each dialogue example:
1. A **buyer LLM** generates an opening offer given the item listing.
2. A **seller LLM** responds to that offer — this is the reference response.
3. CG compares how differently the seller's next-token distributions look when it reads the buyer's message as:
   - **Text channel**: the raw decoded text embedded in the prompt (Arm A).
   - **Latent channel**: the buyer's final-layer hidden states injected at `inject_layer`.

Both channels are scored via **teacher forcing** on the exact same seller response, so the
number reflects pure channel difference — not autoregressive prefix drift.

$$
\text{CG} = \frac{1}{T} \sum_{t=1}^T \bigl[\beta \cdot \mathrm{JS}_t + \alpha \cdot (1 - \cos_t)\bigr]
$$

where $\mathrm{JS}_t \in [0,1]$ (bits) and $1-\cos_t \in [0,2]$.

## What is disabled

| Component | Status | Reason |
|-----------|--------|--------|
| Arm B (editor selfhs) | **OFF** | requires extra `generate()`, not needed for CG |
| Arm C (writer HS generate) | **OFF** | CG rebuilds injection internally with teacher forcing |
| Arm D (writer self-probe) | **OFF** | sanity check arm, not needed for metric |
| Arm A `generate()` | **ON** | needed to produce the reference seller response |
| CG forward passes (×2) | **ON** | core metric |

**Per-example cost: 2 `generate()` calls + 2 fast forward passes.**

## 0 — Install

In [ ]:
# Install the package and ML dependencies
!pip install -q git+https://github.com/YOUR_USERNAME/reagent.git
# Qwen 3.5 users: also run the cell below for full-speed linear-attention kernels
# !pip install -q flash-linear-attention causal-conv1d
# Pin datasets to 2.x — the craigslist_bargains dataset uses a loading script
# which datasets 3.x dropped support for.
!pip install -q matplotlib "datasets>=2.14.0,<3.0.0"

## 1 — Imports & configuration

In [ ]:
import json
import math
import pathlib
import random
import time
import warnings

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import torch

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("[warn] pandas not found — tabular display will be basic. Run: pip install pandas")

from datasets import load_dataset

from selfie_k2v2 import ModelBackend, make_probe_nodes

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION  ← edit here
# ─────────────────────────────────────────────────────────────────────────────

# Model to use as BOTH buyer (writer) and seller (editor).
# A single small model keeps VRAM low while still producing meaningful hidden states.
MODEL_ID         = "Qwen/Qwen3.5-27B"  # swap for any supported model
DTYPE            = torch.bfloat16
DEVICE_MAP       = "auto"

# How many CraigslistBargains examples to evaluate (train split).
# 30 is a good balance of statistical coverage vs. runtime on a single GPU.
N_SAMPLES        = 30
RANDOM_SEED      = 42

# Generation budgets (keep short — we want clean 1-2 sentence outputs)
MAX_BUYER_TOKENS  = 60    # buyer opening message
MAX_SELLER_TOKENS = 60    # seller response (also the CG reference response)

# Injection layer for the latent channel — will be swept across N_LAYER_SWEEP values
# (computed as evenly-spaced indices from 0 → num_layers-1 after model load)
N_LAYER_SWEEP    = 5     # number of layers to compare

# CG weights
CG_ALPHA         = 0.5    # weight on 1 - cosine
CG_BETA          = 0.5    # weight on JS (in bits, already ∈ [0,1])

# Output directory for saved results
OUTPUT_DIR       = pathlib.Path("cg_craigslist_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("Config ready.")
print(f"  model        : {MODEL_ID}")
print(f"  N_SAMPLES    : {N_SAMPLES}")
print(f"  N_LAYER_SWEEP: {N_LAYER_SWEEP}")
print(f"  CG weights   : alpha={CG_ALPHA}, beta={CG_BETA}")

## 2 — Load CraigslistBargains dataset

In [ ]:
# trust_remote_code=True is required because this dataset uses a custom loading script.
# datasets 3.x dropped that support, so the install cell pins to datasets 2.x.
ds = load_dataset("stanfordnlp/craigslist_bargains", trust_remote_code=True)
train_ds = ds["train"]
print(f"Train set: {len(train_ds)} examples")

# Sample N_SAMPLES indices uniformly across train set
all_indices = list(range(len(train_ds)))
random.shuffle(all_indices)
sample_indices = sorted(all_indices[:N_SAMPLES])
sample = [train_ds[i] for i in sample_indices]

# Category distribution in sample
from collections import Counter
cats = [ex["items"]["Category"][0] for ex in sample]
cat_counts = Counter(cats)
print(f"\nSampled {N_SAMPLES} examples. Category breakdown:")
for cat, cnt in sorted(cat_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat:<15s} {cnt}")

# Preview one example
ex = sample[0]
print("\nSample example [0]:")
print(f"  Category  : {ex['items']['Category'][0]}")
print(f"  Title     : {ex['items']['Title'][0][:70]}")
print(f"  Price     : ${ex['items']['Price'][0]:.0f}")
roles = ex['agent_info']['Role']
targets = ex['agent_info']['Target']
buyer_idx = roles.index('buyer')
print(f"  Buyer tgt : ${targets[buyer_idx]:.0f}")
print(f"  First utterance: {ex['utterance'][0][:120]}")

## 3 — Load model

In [ ]:
t0 = time.time()
backend = ModelBackend(
    model_name=MODEL_ID,
    dtype=DTYPE,
    device_map=DEVICE_MAP,
    chat_template_kwargs={"enable_thinking": False},  # disable <think> blocks — faster + cleaner outputs
)
print(f"Model loaded in {time.time()-t0:.1f}s")
print(f"  hidden_size : {backend.hidden_size}")
print(f"  num_layers  : {backend.num_layers}")
print(f"  device      : {backend.device}")
print(f"  strip_think : {backend._strip_thinking}")

# Compute N_LAYER_SWEEP evenly-spaced injection layers (0 to num_layers-1 inclusive)
INJECT_LAYERS = sorted(set(
    int(round(x))
    for x in np.linspace(0, backend.num_layers - 1, N_LAYER_SWEEP)
))
print(f"\n  Layer sweep : {INJECT_LAYERS}  ({len(INJECT_LAYERS)} layers)")

# Both buyer and seller use the same model
writer_backend = editor_backend = backend

## 4 — Build CG probe nodes — one per injection layer

We instantiate **one `probe_comm_gap` closure per layer** using `make_probe_nodes`.
`probe_editor` (Arm A) is shared — it runs once per example, independent of layer.

| Node | Count | Per-example cost |
|------|-------|-----------------|
| `probe_editor` | 1 shared | 1 `generate()` call |
| `probe_comm_gap` × N layers | N closures | N × 2 fast forward passes |

Arms B, C, D are **never instantiated**.

We also provide **domain-specific prompt templates** so the model acts as a seller
rather than an editorial critic:
- `SELLER_USER_TEXT` — text channel (Arm A): frames the buyer message as an offer to reply to
- `SELLER_INJECT_USER_TEXT` — latent channel (CG): same framing, inject marker instead of draft marker

In [ ]:
# ── Domain-specific prompts for negotiation ──────────────────────────────────
# System: tells the model it is acting as a seller
SELLER_SYSTEM = (
    "You are a seller on Craigslist responding to a buyer's opening offer. "
    "Be concise and realistic. Reply in 1-2 sentences."
)

# Text-channel user template (Arm A & CG text channel).
# Must contain exactly one <<<DRAFT_TOKENS>>> marker — the buyer's decoded text
# is spliced here.
SELLER_USER_TEXT = (
    "A buyer sent you the following offer:\n"
    "<OFFER><<<DRAFT_TOKENS>>></OFFER>\n"
    "Your reply as the seller:"
)

# Injection-channel user template (CG latent channel).
# Must contain exactly one <<<INJECT_HERE>>> marker — the buyer's hidden states
# are injected at that position in the residual stream.
SELLER_INJECT_USER_TEXT = (
    "A buyer sent you the following offer:\n"
    "<OFFER><<<INJECT_HERE>>></OFFER>\n"
    "Your reply as the seller:"
)

# ── Arm A node (shared across all layers) ─────────────────────────────────────
_base_nodes = make_probe_nodes(
    writer_backend=writer_backend,
    editor_backend=editor_backend,
    max_editor_tokens=MAX_SELLER_TOKENS,
    inject_layer=0,                        # doesn't matter for probe_editor
    editor_system=SELLER_SYSTEM,
    editor_user_text=SELLER_USER_TEXT,
    editor_inject_user_text=SELLER_INJECT_USER_TEXT,
    run_comm_gap=False,                    # skip CG here; build per-layer below
)
probe_editor = _base_nodes["probe_editor"]

# ── One probe_comm_gap closure per injection layer ────────────────────────────
comm_gap_nodes = {}          # {layer_idx: callable}
for layer in INJECT_LAYERS:
    _nodes = make_probe_nodes(
        writer_backend=writer_backend,
        editor_backend=editor_backend,
        max_editor_tokens=MAX_SELLER_TOKENS,
        inject_layer=layer,
        editor_system=SELLER_SYSTEM,
        editor_user_text=SELLER_USER_TEXT,
        editor_inject_user_text=SELLER_INJECT_USER_TEXT,
        run_comm_gap=True,
        cg_alpha=CG_ALPHA,
        cg_beta=CG_BETA,
    )
    comm_gap_nodes[layer] = _nodes["probe_comm_gap"]

print(f"probe_editor    : 1 shared node (Arm A)")
print(f"probe_comm_gap  : {len(comm_gap_nodes)} nodes — layers {INJECT_LAYERS}")
print(f"Arms B, C, D    : NOT instantiated")

## 5 — Negotiation prompt builders

In [ ]:
def parse_scenario(example: dict) -> dict:
    """Extract negotiation scenario info from a CraigslistBargains example."""
    roles   = example["agent_info"]["Role"]
    targets = example["agent_info"]["Target"]
    buyer_idx  = roles.index("buyer")
    seller_idx = roles.index("seller")
    return {
        "category"     : example["items"]["Category"][0],
        "title"        : example["items"]["Title"][0],
        "description"  : example["items"]["Description"][0],
        "listing_price": float(example["items"]["Price"][0]),
        "buyer_target" : float(targets[buyer_idx]),
        "seller_target": float(targets[seller_idx]),
        "first_intent" : (example["dialogue_acts"]["intent"] or ["unknown"])[0],
        "n_turns"      : len(example["utterance"]),
        "human_opening": example["utterance"][0] if example["utterance"] else "",
    }


def buyer_system_prompt(sc: dict) -> str:
    return (
        "You are a buyer on Craigslist. "
        "Write a short, realistic opening offer in 1-2 sentences."
    )


def buyer_user_prompt(sc: dict) -> str:
    desc = sc["description"][:200].rstrip()
    return (
        f"Item: {sc['title']} (Category: {sc['category']})\n"
        f"Description: {desc}\n"
        f"Listing price: ${sc['listing_price']:.0f}\n"
        f"Your target price: ${sc['buyer_target']:.0f}\n\n"
        f"Write your opening message to the seller:"
    )


# Smoke test
sc0 = parse_scenario(sample[0])
print("Scenario 0:")
for k, v in sc0.items():
    print(f"  {k:<16}: {str(v)[:80]}")
print("\nBuyer user prompt (first 300 chars):")
print(buyer_user_prompt(sc0)[:300])

## 6 — Run CG evaluation over sampled dataset

For each example:
1. **Buyer generates** an opening offer (hidden states captured).
2. **Seller responds** (Arm A — produces the reference seller response).
3. For **each of the N injection layers**: CG is computed via 2 fast teacher-forced forward passes.

Per-example cost: **2 `generate()` calls + N×2 fast forward passes** (N = number of layers in sweep).

In [ ]:
try:
    from tqdm.auto import tqdm
    _use_tqdm = True
except ImportError:
    _use_tqdm = False

results = []           # list of per-example result dicts (scalars + metadata)
raw_tensors = []       # list of per-example tensor dicts (per-step data per layer)

iter_samples = tqdm(enumerate(sample), total=len(sample)) if _use_tqdm else enumerate(sample)

for i, example in iter_samples:
    sc = parse_scenario(example)
    t_start = time.time()

    try:
        # ── Step 1: Buyer generates opening offer ──────────────────────────────
        buyer_prompt_ids = writer_backend.build_chat_prompt(
            system=buyer_system_prompt(sc),
            user=buyer_user_prompt(sc),
        )
        buyer_result = writer_backend.generate(
            prompt_ids=buyer_prompt_ids,
            max_new_tokens=MAX_BUYER_TOKENS,
            capture_hidden=True,    # required for latent channel
        )

        if not buyer_result.gen_token_ids:
            print(f"[{i}] SKIP — buyer produced no tokens.")
            continue

        # ── Step 2: Seller responds (Arm A — shared across all layers) ─────────
        state = {
            "writer_result"          : buyer_result,
            "writer_output_text"     : buyer_result.output_text,
            "writer_output_token_ids": buyer_result.gen_token_ids,
        }
        state = probe_editor(state)   # adds editor_result, editor_prompt_ids, editor_verdict

        if not state.get("editor_verdict", "").strip():
            print(f"[{i}] SKIP — seller produced no text.")
            continue

        # ── Step 3: CG for each injection layer ────────────────────────────────
        cg_by_layer = {}
        tensors_by_layer = {}

        for layer, cg_node in comm_gap_nodes.items():
            state_l = cg_node(state)
            cg = state_l["comm_gap"]
            if cg.get("T", 0) > 0:
                cg_by_layer[layer] = cg
                tensors_by_layer[layer] = {
                    "per_step"    : cg["per_step"],
                    "JS_per_step" : cg["JS_per_step"],
                    "COS_per_step": cg["COS_per_step"],
                }

        if not cg_by_layer:
            print(f"[{i}] SKIP — all layer CG computations failed.")
            continue

        elapsed = time.time() - t_start

        # ── Scalar record ──────────────────────────────────────────────────────
        rec = {
            "idx"           : i,
            "category"      : sc["category"],
            "listing_price" : sc["listing_price"],
            "buyer_target"  : sc["buyer_target"],
            "price_ratio"   : sc["buyer_target"] / sc["listing_price"] if sc["listing_price"] > 0 else None,
            "first_intent"  : sc["first_intent"],
            "n_turns"       : sc["n_turns"],
            "buyer_tokens"  : len(buyer_result.gen_token_ids),
            "seller_tokens" : next(iter(cg_by_layer.values()))["T"],
            # Per-layer CG scalars
            "CG_by_layer"   : {l: cg_by_layer[l]["CG"]      for l in cg_by_layer},
            "JS_by_layer"   : {l: cg_by_layer[l]["JS_mean"]  for l in cg_by_layer},
            "COS_by_layer"  : {l: cg_by_layer[l]["COS_mean"] for l in cg_by_layer},
            # Convenience: mean CG across all layers (for sorting)
            "CG_mean"       : float(np.mean([cg_by_layer[l]["CG"] for l in cg_by_layer])),
            "buyer_text"    : buyer_result.output_text,
            "seller_text"   : state["editor_verdict"],
            "elapsed_s"     : elapsed,
        }
        results.append(rec)
        raw_tensors.append({"idx": i, "by_layer": tensors_by_layer})

        if not _use_tqdm:
            layer_str = "  ".join(
                f"L{l}:{cg_by_layer[l]['CG']:.3f}" for l in INJECT_LAYERS if l in cg_by_layer
            )
            print(f"[{i:3d}/{len(sample)}] {layer_str}  cat={sc['category']}  ({elapsed:.1f}s)")

    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"[{i}] ERROR: {e}")
        continue

print(f"\nCompleted {len(results)}/{len(sample)} examples.")

## 7 — Results DataFrame & summary statistics

In [ ]:
if HAS_PANDAS:
    # Flatten per-layer CG into columns CG_L0, CG_L8, etc.
    rows = []
    for r in results:
        row = {k: v for k, v in r.items() if k not in ("CG_by_layer", "JS_by_layer", "COS_by_layer")}
        for l in INJECT_LAYERS:
            row[f"CG_L{l}"]  = r["CG_by_layer"].get(l, float("nan"))
            row[f"JS_L{l}"]  = r["JS_by_layer"].get(l, float("nan"))
            row[f"COS_L{l}"] = r["COS_by_layer"].get(l, float("nan"))
        rows.append(row)
    df = pd.DataFrame(rows)

    cg_cols = [f"CG_L{l}" for l in INJECT_LAYERS]
    print(df[["category", "buyer_tokens", "price_ratio"] + cg_cols].to_string(index=False))
    print()
    print("── Per-layer CG summary ──")
    print(df[cg_cols].describe().round(4))
else:
    print("Per-layer mean CG:")
    for l in INJECT_LAYERS:
        vals = [r["CG_by_layer"].get(l, float("nan")) for r in results]
        vals = [v for v in vals if not (isinstance(v, float) and v != v)]
        print(f"  layer {l:3d}: mean={np.mean(vals):.4f}  std={np.std(vals):.4f}")

## 8 — Plots

In [ ]:
# ─── Plot 1: CG distribution per layer ───────────────────────────────────────
fig, axes = plt.subplots(1, len(INJECT_LAYERS), figsize=(3.5 * len(INJECT_LAYERS), 4),
                          sharey=True)
if len(INJECT_LAYERS) == 1:
    axes = [axes]

layer_colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(INJECT_LAYERS)))

for ax, layer, color in zip(axes, INJECT_LAYERS, layer_colors):
    vals = [r["CG_by_layer"].get(layer, float("nan")) for r in results]
    vals = [v for v in vals if not (v != v)]  # drop NaN
    ax.hist(vals, bins=10, color=color, alpha=0.85, edgecolor="white")
    ax.axvline(np.mean(vals), color="black", linestyle="--", linewidth=1.5,
               label=f"mean={np.mean(vals):.3f}")
    ax.set_title(f"Layer {layer}", fontsize=10)
    ax.set_xlabel("CG")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

axes[0].set_ylabel("count")
fig.suptitle(
    f"CG distribution per injection layer — {len(results)} examples\n"
    f"Model: {MODEL_ID}",
    fontsize=11,
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot1_cg_distribution_by_layer.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot1_cg_distribution_by_layer.png")

In [ ]:
# ─── Plot 2: CG by product category (mean across layers) ─────────────────────
from collections import defaultdict

cat_cg = defaultdict(list)
for r in results:
    cat_cg[r["category"]].append(r["CG_mean"])

categories = sorted(cat_cg.keys(), key=lambda c: np.median(cat_cg[c]), reverse=True)
data_by_cat = [cat_cg[c] for c in categories]

fig, ax = plt.subplots(figsize=(max(6, len(categories) * 1.3), 5))
bp = ax.boxplot(
    data_by_cat,
    labels=categories,
    patch_artist=True,
    medianprops=dict(color="black", linewidth=2),
)
colors = plt.cm.Set2(np.linspace(0, 1, len(categories)))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

for j, (vals, cat) in enumerate(zip(data_by_cat, categories), start=1):
    jitter = np.random.uniform(-0.15, 0.15, len(vals))
    ax.scatter(j + jitter, vals, color="black", s=18, alpha=0.6, zorder=3)

ax.set_xlabel("Product category")
ax.set_ylabel("Communication Gap — mean across layers")
ax.set_title(
    "CG by product category (mean CG across all sweep layers)\n"
    "Higher CG → more info lost when buyer's text replaces latent representation"
)
ax.grid(axis="y", alpha=0.25)
for j, (vals, cat) in enumerate(zip(data_by_cat, categories), start=1):
    ax.text(j, ax.get_ylim()[0] * 0.98, f"n={len(vals)}", ha="center",
            va="bottom", fontsize=8, color="gray")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot2_cg_by_category.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot2_cg_by_category.png")

In [ ]:
# ─── Plot 3: JS vs COS per layer (mean across examples) ──────────────────────
# Scatter: each point = one example at one layer; colour = layer depth
fig, ax = plt.subplots(figsize=(7, 6))

colors_cat = {cat: plt.cm.tab10(i) for i, cat in enumerate(sorted(cat_cg.keys()))}
markers = ["o", "s", "^", "D", "P"]

for li, layer in enumerate(INJECT_LAYERS):
    for r in results:
        js  = r["JS_by_layer"].get(layer, float("nan"))
        cos = r["COS_by_layer"].get(layer, float("nan"))
        if js != js or cos != cos:
            continue
        ax.scatter(js, cos,
                   color=layer_colors[li],
                   marker=markers[li % len(markers)],
                   s=45, alpha=0.7, edgecolors="white", linewidths=0.4,
                   label=f"Layer {layer}")

# Diagonal line: equal CG contribution from both terms
lim = max(
    max(r["JS_by_layer"].get(l, 0) for r in results for l in INJECT_LAYERS if r["JS_by_layer"].get(l, 0)==r["JS_by_layer"].get(l, 0)),
    max(r["COS_by_layer"].get(l, 0) for r in results for l in INJECT_LAYERS if r["COS_by_layer"].get(l, 0)==r["COS_by_layer"].get(l, 0)),
) * 1.05
ax.plot([0, lim * CG_ALPHA / max(CG_BETA, 1e-9)], [0, lim], "--",
        color="gray", alpha=0.5, linewidth=1, label="equal contribution (β·JS = α·COS)")

# Deduplicate legend
seen = set()
handles, labels = ax.get_legend_handles_labels()
unique = [(h, l) for h, l in zip(handles, labels) if l not in seen and not seen.add(l)]
ax.legend(*zip(*unique), fontsize=8, loc="upper left")

ax.set_xlabel(f"JS divergence (mean, bits) — β={CG_BETA}")
ax.set_ylabel(f"1 − cosine similarity (mean) — α={CG_ALPHA}")
ax.set_title(
    "JS vs. 1-cos per example × layer\n"
    "Each symbol = one injection layer depth"
)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot3_js_vs_cos_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot3_js_vs_cos_scatter.png")

In [ ]:
# ─── Plot 4: CG_mean vs buyer message length + price aggressiveness ──────────
buyer_lens   = [r["buyer_tokens"] for r in results]
cg_mean_vals = [r["CG_mean"]     for r in results]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: scatter with regression line
ax = axes[0]
ax.scatter(buyer_lens, cg_mean_vals,
           c=[colors_cat[r["category"]] for r in results],
           s=55, alpha=0.8, edgecolors="white", linewidths=0.5)
z = np.polyfit(buyer_lens, cg_mean_vals, 1)
p = np.poly1d(z)
xs = np.linspace(min(buyer_lens), max(buyer_lens), 50)
ax.plot(xs, p(xs), "r--", linewidth=1.5, alpha=0.7, label=f"slope = {z[0]:.4f}")
corr = np.corrcoef(buyer_lens, cg_mean_vals)[0, 1]
ax.set_xlabel("Buyer message length (tokens)")
ax.set_ylabel("CG (mean across layers)")
ax.set_title(f"CG vs. buyer message length\nPearson r = {corr:.3f}")
ax.legend(fontsize=9)
ax.grid(alpha=0.25)

# Right: CG vs price aggressiveness
ax = axes[1]
price_ratios  = [r["price_ratio"] for r in results if r["price_ratio"] is not None]
cg_for_ratio  = [r["CG_mean"]     for r in results if r["price_ratio"] is not None]
cat_for_ratio = [r["category"]    for r in results if r["price_ratio"] is not None]
ax.scatter(price_ratios, cg_for_ratio,
           c=[colors_cat[c] for c in cat_for_ratio],
           s=55, alpha=0.8, edgecolors="white", linewidths=0.5)
if len(price_ratios) > 2:
    z2 = np.polyfit(price_ratios, cg_for_ratio, 1)
    p2 = np.poly1d(z2)
    xs2 = np.linspace(min(price_ratios), max(price_ratios), 50)
    ax.plot(xs2, p2(xs2), "r--", linewidth=1.5, alpha=0.7)
    corr2 = np.corrcoef(price_ratios, cg_for_ratio)[0, 1]
    ax.set_title(f"CG vs. buyer aggressiveness (target/listing)\nPearson r = {corr2:.3f}")
ax.axvline(1.0, color="gray", linestyle=":", alpha=0.6, label="target = listing")
ax.set_xlabel("Price ratio (buyer target / listing price)")
ax.set_ylabel("CG (mean across layers)")
ax.legend(fontsize=9)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot4_cg_vs_length_and_price.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot4_cg_vs_length_and_price.png")

In [ ]:
# ─── Plot 5: Per-step CG heatmap — highest vs lowest CG examples ─────────────
# Use the middle layer for the per-step comparison
ref_layer = INJECT_LAYERS[len(INJECT_LAYERS) // 2]

sorted_by_cg = sorted(
    [(r["CG_mean"], j) for j, r in enumerate(results)], reverse=True
)
N_SHOW = min(5, len(results))
top_idxs = [j for _, j in sorted_by_cg[:N_SHOW]]
bot_idxs = [j for _, j in sorted_by_cg[-N_SHOW:]]

def pad_series(series_list):
    max_len = max(len(s) for s in series_list)
    out = np.full((len(series_list), max_len), np.nan)
    for i, s in enumerate(series_list):
        out[i, :len(s)] = s
    return out

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

for ax, idxs, title_prefix in zip(axes,
                                    [top_idxs, bot_idxs],
                                    [f"Top-{N_SHOW} highest CG (mean)",
                                     f"Bottom-{N_SHOW} lowest CG (mean)"]):
    row_labels = [
        f"[{results[j]['idx']}] {results[j]['category']}  mean={results[j]['CG_mean']:.3f}"
        for j in idxs
    ]
    series = [
        raw_tensors[j]["by_layer"].get(ref_layer, {}).get("per_step",
            torch.zeros(1)).numpy()
        for j in idxs
    ]
    mat = pad_series(series)

    cmap = plt.cm.viridis.copy(); cmap.set_bad("white")
    masked = np.ma.masked_invalid(mat)
    im = ax.imshow(masked, aspect="auto", cmap=cmap, vmin=0,
                   vmax=max(np.nanmax(mat), 1e-6))
    ax.set_yticks(range(len(idxs)))
    ax.set_yticklabels(row_labels, fontsize=8)
    ax.set_xlabel(f"Seller response token position (ref_layer={ref_layer})")
    ax.set_title(f"{title_prefix} — per-step CG at injection layer {ref_layer}")
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="CG per step")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot5_perstep_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot5_perstep_heatmap.png")

In [ ]:
# ─── Plot 6: Per-step JS vs COS breakdown for top/bottom 3 (at ref_layer) ────
N_LINE = min(3, len(results))
top3  = [j for _, j in sorted_by_cg[:N_LINE]]
bot3  = [j for _, j in sorted_by_cg[-N_LINE:]]

fig, axes = plt.subplots(N_LINE, 2, figsize=(14, 3.5 * N_LINE),
                          sharex=False, sharey=False)
if N_LINE == 1:
    axes = axes[np.newaxis, :]

for row, (hi, lo) in enumerate(zip(top3, bot3)):
    for col, j in enumerate([hi, lo]):
        ax  = axes[row, col]
        r   = results[j]
        rt  = raw_tensors[j]["by_layer"].get(ref_layer, {})
        T   = r["seller_tokens"]
        xs  = range(T)

        ps  = rt.get("per_step",    torch.zeros(T)).numpy()[:T]
        js  = rt.get("JS_per_step", torch.zeros(T)).numpy()[:T]
        cos = rt.get("COS_per_step",torch.zeros(T)).numpy()[:T]

        ax.fill_between(xs, js  * CG_BETA,  alpha=0.45, color="coral",
                        label=f"β·JS (β={CG_BETA})")
        ax.fill_between(xs, cos * CG_ALPHA, alpha=0.45, color="steelblue",
                        label=f"α·(1-cos) (α={CG_ALPHA})")
        ax.plot(xs, ps, color="black", linewidth=1.5, label="CG per step")
        ax.axhline(r["CG_by_layer"].get(ref_layer, 0), color="gray",
                   linestyle=":", alpha=0.7,
                   label=f"mean CG(L{ref_layer})={r['CG_by_layer'].get(ref_layer,0):.3f}")
        side = "HIGH CG" if col == 0 else "LOW CG"
        ax.set_title(
            f"{side} | [{r['idx']}] cat={r['category']}  price=${r['listing_price']:.0f}\n"
            f"Buyer: \"{r['buyer_text'][:60].strip()}...\"",
            fontsize=8,
        )
        ax.set_xlabel("token position")
        ax.set_ylabel("distance")
        ax.legend(fontsize=7, loc="upper right")
        ax.grid(alpha=0.2)

fig.suptitle(
    f"Per-step CG breakdown at injection layer {ref_layer}\n"
    "JS (distributional) vs. 1-cos (directional)", fontsize=11
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot6_perstep_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot6_perstep_breakdown.png")

## 9 — Layer sweep analysis

These four plots directly answer: **does injection depth change the Communication Gap, and how?**

In [ ]:
# ─── Layer plot A: mean CG ± std across layers (with individual example lines) ─
cg_means = []
cg_stds  = []
for l in INJECT_LAYERS:
    vals = [r["CG_by_layer"][l] for r in results if l in r["CG_by_layer"]]
    cg_means.append(np.mean(vals))
    cg_stds.append(np.std(vals))

fig, ax = plt.subplots(figsize=(8, 5))

# Individual example lines (light, behind)
for r in results:
    ys = [r["CG_by_layer"].get(l, float("nan")) for l in INJECT_LAYERS]
    ax.plot(INJECT_LAYERS, ys, color="steelblue", alpha=0.15, linewidth=1)

# Mean ± std band
ax.fill_between(
    INJECT_LAYERS,
    np.array(cg_means) - np.array(cg_stds),
    np.array(cg_means) + np.array(cg_stds),
    alpha=0.25, color="steelblue", label="mean ± 1 std"
)
ax.plot(INJECT_LAYERS, cg_means, "o-", color="steelblue",
        linewidth=2.5, markersize=8, label="mean CG")

ax.set_xlabel("Injection layer")
ax.set_ylabel("Communication Gap (CG)")
ax.set_title(
    "CG vs. injection layer depth\n"
    "Low layer = inject near embedding; high = inject near output"
)
ax.set_xticks(INJECT_LAYERS)
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot7_cg_vs_layer.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot7_cg_vs_layer.png")

In [ ]:
# ─── Layer plot B: example × layer heatmap ────────────────────────────────────
# rows = examples sorted by mean CG; cols = injection layers
sorted_results = sorted(results, key=lambda r: r["CG_mean"], reverse=True)

mat = np.array([
    [r["CG_by_layer"].get(l, float("nan")) for l in INJECT_LAYERS]
    for r in sorted_results
])
row_labels = [
    f"[{r['idx']}] {r['category']}"
    for r in sorted_results
]

fig, ax = plt.subplots(figsize=(max(5, len(INJECT_LAYERS) * 1.1), max(4, len(results) * 0.35)))
cmap = plt.cm.viridis.copy(); cmap.set_bad("lightgray")
masked = np.ma.masked_invalid(mat)
im = ax.imshow(masked, aspect="auto", cmap=cmap)
ax.set_xticks(range(len(INJECT_LAYERS)))
ax.set_xticklabels([f"L{l}" for l in INJECT_LAYERS])
ax.set_yticks(range(len(sorted_results)))
ax.set_yticklabels(row_labels, fontsize=7)
ax.set_xlabel("Injection layer")
ax.set_title("CG heatmap: example (sorted by mean CG) × injection layer")

# Annotate cells
for ri in range(mat.shape[0]):
    for ci in range(mat.shape[1]):
        v = mat[ri, ci]
        if v == v:  # not NaN
            ax.text(ci, ri, f"{v:.2f}", ha="center", va="center",
                    fontsize=7, color="white" if v > np.nanmax(mat) * 0.5 else "black")

plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="CG")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot8_cg_layer_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot8_cg_layer_heatmap.png")

In [ ]:
# ─── Layer plot C: CG by category × layer (grouped bars) ────────────────────
all_cats = sorted(set(r["category"] for r in results))
n_cats   = len(all_cats)
n_layers = len(INJECT_LAYERS)
bar_w    = 0.8 / n_layers
x_base   = np.arange(n_cats)

fig, ax = plt.subplots(figsize=(max(7, n_cats * 1.5), 5))

for li, (layer, color) in enumerate(zip(INJECT_LAYERS, layer_colors)):
    means = []
    stds  = []
    for cat in all_cats:
        vals = [r["CG_by_layer"].get(layer, float("nan"))
                for r in results if r["category"] == cat]
        vals = [v for v in vals if v == v]
        means.append(np.mean(vals) if vals else 0)
        stds.append(np.std(vals)  if len(vals) > 1 else 0)
    offset = (li - n_layers / 2 + 0.5) * bar_w
    bars = ax.bar(x_base + offset, means, bar_w * 0.9,
                  color=color, alpha=0.85, label=f"L{layer}")
    ax.errorbar(x_base + offset, means, stds,
                fmt="none", color="black", capsize=3, linewidth=1)

ax.set_xticks(x_base)
ax.set_xticklabels(all_cats, rotation=20, ha="right")
ax.set_xlabel("Product category")
ax.set_ylabel("Mean CG")
ax.set_title("Mean CG per category × injection layer\n(error bars = ±1 std)")
ax.legend(title="Inject layer", fontsize=8)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot9_cg_category_layer.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot9_cg_category_layer.png")

In [ ]:
# ─── Layer plot D: per-step CG at each layer for a single example ─────────────
# Pick the example with the highest mean CG as the showcase
showcase_j = sorted_by_cg[0][1]
r_show     = results[showcase_j]
rt_show    = raw_tensors[showcase_j]["by_layer"]

fig, ax = plt.subplots(figsize=(10, 5))
T_max = r_show["seller_tokens"]

for li, (layer, color) in enumerate(zip(INJECT_LAYERS, layer_colors)):
    if layer not in rt_show:
        continue
    ps = rt_show[layer]["per_step"].numpy()[:T_max]
    xs = range(len(ps))
    ax.plot(xs, ps, color=color, linewidth=2, label=f"Layer {layer}  (mean={r_show['CG_by_layer'].get(layer,0):.3f})")

ax.set_xlabel("Seller response token position")
ax.set_ylabel("CG per step")
ax.set_title(
    f"Per-step CG at each injection layer — example [{r_show['idx']}]  "
    f"cat={r_show['category']}\n"
    f"Buyer: \"{r_show['buyer_text'][:80].strip()}\""
)
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plot10_perstep_layer_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot10_perstep_layer_sweep.png")

## 10 — Additional probing: inspect high vs low CG examples

Reading the actual buyer and seller texts for the extreme examples lets you
form qualitative hypotheses about what drives the gap.

In [ ]:
BAR  = "═" * 72
THIN = "─" * 72
N_INSPECT = min(5, len(results))

print(BAR)
print(f"  TOP-{N_INSPECT} HIGHEST CG  (latent ≠ text — large communication gap)")
print(BAR)
for rank, (cg_val, j) in enumerate(sorted_by_cg[:N_INSPECT], 1):
    r = results[j]
    layer_str = "  ".join(f"L{l}:{r['CG_by_layer'].get(l,0):.3f}" for l in INJECT_LAYERS)
    print(f"#{rank}  idx={r['idx']}  mean_CG={r['CG_mean']:.4f}  cat={r['category']}")
    print(f"   layers: {layer_str}")
    print(f"   listing=${r['listing_price']:.0f}  buyer_target=${r['buyer_target']:.0f}  "
          f"ratio={r['price_ratio']:.2f}")
    print(f"   BUYER  : {r['buyer_text'].strip()[:140]}")
    print(f"   SELLER : {r['seller_text'].strip()[:140]}")
    print(THIN)

print()
print(BAR)
print(f"  BOTTOM-{N_INSPECT} LOWEST CG  (latent ≈ text — channels agree)")
print(BAR)
for rank, (cg_val, j) in enumerate(sorted_by_cg[-N_INSPECT:][::-1], 1):
    r = results[j]
    layer_str = "  ".join(f"L{l}:{r['CG_by_layer'].get(l,0):.3f}" for l in INJECT_LAYERS)
    print(f"#{rank}  idx={r['idx']}  mean_CG={r['CG_mean']:.4f}  cat={r['category']}")
    print(f"   layers: {layer_str}")
    print(f"   listing=${r['listing_price']:.0f}  buyer_target=${r['buyer_target']:.0f}  "
          f"ratio={r['price_ratio']:.2f}")
    print(f"   BUYER  : {r['buyer_text'].strip()[:140]}")
    print(f"   SELLER : {r['seller_text'].strip()[:140]}")
    print(THIN)

## 11 — Save all outputs

In [ ]:
# ── 1. Scalar results → JSON ──────────────────────────────────────────────────
def _json_safe(v):
    if isinstance(v, (np.floating, float)):  return float(v)
    if isinstance(v, (np.integer, int)):     return int(v)
    if isinstance(v, dict):                  return {str(k): _json_safe(vv) for k, vv in v.items()}
    return v

scalar_path = OUTPUT_DIR / "cg_results.json"
scalar_out  = [{k: _json_safe(v) for k, v in r.items()} for r in results]
with open(scalar_path, "w", encoding="utf-8") as f:
    json.dump(scalar_out, f, indent=2, ensure_ascii=False)
print(f"Scalar results  → {scalar_path}  ({len(scalar_out)} records)")

# ── 2. Per-step tensors → torch save ──────────────────────────────────────────
tensor_path = OUTPUT_DIR / "cg_perstep_tensors.pt"
torch.save(raw_tensors, tensor_path)
print(f"Per-step tensors → {tensor_path}  ({len(raw_tensors)} records)")

# ── 3. Config snapshot ────────────────────────────────────────────────────────
config_path = OUTPUT_DIR / "config.json"
config_snap = {
    "model_id"       : MODEL_ID,
    "dtype"          : str(DTYPE),
    "n_samples"      : N_SAMPLES,
    "random_seed"    : RANDOM_SEED,
    "max_buyer_tok"  : MAX_BUYER_TOKENS,
    "max_seller_tok" : MAX_SELLER_TOKENS,
    "inject_layers"  : INJECT_LAYERS,
    "cg_alpha"       : CG_ALPHA,
    "cg_beta"        : CG_BETA,
    "n_completed"    : len(results),
    "hidden_size"    : backend.hidden_size,
    "num_layers"     : backend.num_layers,
}
with open(config_path, "w") as f:
    json.dump(config_snap, f, indent=2)
print(f"Config snapshot  → {config_path}")

# ── 4. CSV (if pandas) ────────────────────────────────────────────────────────
if HAS_PANDAS:
    csv_path = OUTPUT_DIR / "cg_results.csv"
    df.to_csv(csv_path, index=False)
    print(f"CSV              → {csv_path}")

print()
print("=== Reload example ===")
print(f"import json, torch")
print(f"results     = json.load(open('{scalar_path}'))")
print(f"raw_tensors = torch.load('{tensor_path}')")
print(f"# raw_tensors[i]['by_layer'][layer_idx]['per_step']  → (T,) tensor")

## Interpretation guide

| Pattern | What it suggests |
|---------|------------------|
| **CG ≈ 0 across all examples** | Text and latent channels carry equivalent information for this model/layer. Injection at `inject_layer=0` may be too early — try `num_layers//2`. |
| **CG varies widely across examples** | Methodology is sensitive — some negotiation contexts benefit more from the latent channel than others. |
| **High CG for specific categories** | The model's latent representation of that domain diverges from its decoded text. Latent injection may be particularly valuable there. |
| **JS >> 1-cos** | Channels agree in logit *direction* but disagree on *probability mass* — the model changes which tokens it thinks are likely, not just their relative ordering. |
| **1-cos >> JS** | Logit vectors point differently but the softmax distributions are still similar — scale effects dominate. |
| **CG decreases with price_ratio** | More aggressive (low) offers create larger gaps — the model's latent encoding of aggressive negotiation carries more information than its text output. |
| **Per-step: gap spikes at later tokens** | Information loss accumulates through the seller's response — early tokens agree but later ones diverge as the seller reasons further. |